# SKAT-R: revision experiments (supervisor issues 2 and 3)

This notebook re-uses the **unchanged** data pipeline, models, training recipe and metrics of the original study
(sections 1–4 below are copied from `SKAT-R_notebook.ipynb`) and adds only the experiments requested in the review:

| Issue | What is run here | Datasets |
|---|---|---|
| 2a. Stronger post hoc explainers | **Integrated Gradients** (32 steps, [PAD] baseline) and last-layer **[CLS] attention** for fine-tuned BiomedBERT, next to gradient×input | Symptom2Disease, ADE, Medical Abstracts |
| 2b. A faithfulness check FR does not optimise | **Top-1 decision flip** (does deleting the single top word change the prediction?) and **Spearman correlation with the true leave-one-word-out effect** | same |
| 3. One prior faithfulness method | **Lin-TaSc** (Chrysostomou & Aletras, 2021), re-implemented on the same encoder, recipe and splits | Symptom2Disease, Medical Abstracts |

ERASER comprehensiveness/sufficiency (AOPC) and plausibility (ADE expert spans; Symptom2Disease knowledge base) are also
computed for every new explanation. Same seeds (42, 123, 456, 789, 2024), same leakage-free splits, same explanation
subsets, SKAT-R hyper-parameters as selected in the original study (not re-tuned).

**How to run on Kaggle:** Settings → Accelerator **GPU T4 x1**, **Internet ON** → *Save Version → Save & Run All*.
Datasets run in the order Symptom2Disease → ADE → Medical Abstracts. Errors, if any, show up within the first minutes
(Symptom2Disease is small). If the 12 h session ends early, attach this version's output (*Add data → Your work*) and run
again: finished runs are skipped. Download `skatr_revision_results.zip` from the Output tab at the end and send it back.

**Estimated GPU time (T4):** Symptom2Disease ≈ 0.5 h, ADE ≈ 2 h, Medical Abstracts ≈ 4–5 h (one or two sessions).
To save time, edit `REV_PLAN` in the revision cell (for example, remove `"medical_abstracts"` or `"tasc"`).


In [1]:
# SKATR_SETUP 1
# ============================== CONFIG ==============================
import os, re, json, time, math, glob, random, copy, warnings, platform, sys, shutil, difflib
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# ---------------- DATASET (one Kaggle version per dataset) ----------------
# backbones: SKAT-R AND plain fine-tuning are trained on each (paired comparison);  extra_ft: plain fine-tuning only (baseline)
# "symptom2disease"            : Kaggle niyarrbarman/symptom2disease, 24 diseases, patient-style vignettes
# "mtsamples"                  : Kaggle tboyle10/medicaltranscriptions, specialty classification (real transcriptions)
# "medical_abstracts"          : HF TimSchopf/medical_abstracts, 5 groups, conflicting duplicate abstracts REMOVED (documented)
# "medical_abstracts_official" : same data exactly as published (conflicting duplicates kept) — comparability with literature
# "ade_corpus"                 : HF ade-benchmark-corpus/ade_corpus_v2, ADE vs not, with EXPERT drug/effect spans = human rationales
DATASET = "symptom2disease"

SEEDS = [42, 123, 456, 789, 2024]      # 10 seeds if budget allows: + [7, 99, 314, 1000, 31337]

DISTIL = "distilbert-base-uncased"
CLINICAL = "emilyalsentzer/Bio_ClinicalBERT"
BIOMED = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
BACKBONE_NAMES = {DISTIL: "DistilBERT", CLINICAL: "Bio_ClinicalBERT", BIOMED: "BiomedBERT"}
MAIN_BACKBONE = BIOMED                 # fixed a priori (domain-pretrained encoder); all backbones are reported anyway

RUN_LLM = False   # not needed for the revision experiments
LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"  # open weights, no login needed; fits a T4 in fp16

ALL_ABL = ["w/o FR", "w/o RCR", "w/o RCR & FR", "w/o lexical branch", "FR sufficiency only"]
_DS = {
    "symptom2disease": dict(
        max_len=128, epochs=8, batch_size=16, fixed_test=False, drop_conflicts=True, probes=True,
        backbones=[DISTIL, CLINICAL, BIOMED], extra_ft=[], ablations=ALL_ABL,   # full ablation study lives here
        hp_grid=dict(alpha=[0.10, 0.25, 0.50], beta_s=[0.5, 1.0]), hp_from=None,
        n_explain=200, max_occ_words=64, llm_max_tokens=256,
        task="Identify the most likely disease described in this patient's own description of their symptoms."),
    "mtsamples": dict(
        max_len=512, epochs=6, batch_size=8, fixed_test=False, drop_conflicts=True, probes=False,
        backbones=[DISTIL, BIOMED], extra_ft=[CLINICAL], ablations=["w/o FR", "w/o RCR"],
        hp_grid=dict(alpha=[0.10, 0.25, 0.50], beta_s=[1.0]), hp_from=None,
        n_explain=120, max_occ_words=96, llm_max_tokens=768,
        task="Identify the medical specialty of this clinical transcription."),
    "ade_corpus": dict(   # PRIMARY plausibility dataset: expert-annotated evidence spans, no new annotation needed
        max_len=96, epochs=3, batch_size=16, fixed_test=False, drop_conflicts=True, probes=False,
        backbones=[DISTIL, BIOMED], extra_ft=[CLINICAL], ablations=["w/o FR", "w/o RCR"],
        hp_grid=dict(alpha=[0.10, 0.25, 0.50], beta_s=[1.0]), hp_from=None,
        n_explain=200, explain_pos_neg=(150, 50), max_occ_words=48, llm_max_tokens=128,
        task="Decide whether this sentence from a medical case report describes an adverse effect caused by a drug "
             "(Related) or not (Not-Related)."),
    "medical_abstracts": dict(
        max_len=320, epochs=4, batch_size=16, fixed_test=True, drop_conflicts=True, probes=False,
        backbones=[BIOMED], extra_ft=[DISTIL, CLINICAL], ablations=["w/o FR", "w/o RCR"],
        hp_grid=dict(alpha=[0.10, 0.25], beta_s=[1.0]), hp_from=None,
        n_explain=150, max_occ_words=128, llm_max_tokens=512,
        task="Identify the disease category that this medical abstract is about."),
    "medical_abstracts_official": dict(
        max_len=320, epochs=4, batch_size=16, fixed_test=True, drop_conflicts=False, probes=False,
        backbones=[BIOMED], extra_ft=[CLINICAL], ablations=[],
        hp_grid=dict(alpha=[0.25], beta_s=[1.0]), hp_from="medical_abstracts",   # re-uses the cleaned run's selection
        n_explain=0, max_occ_words=0, llm_max_tokens=512,
        task="Identify the disease category that this medical abstract is about."),
}
DS_CFG = _DS[DATASET]
MTSAMPLES_MIN_PER_CLASS = 50
MTSAMPLES_DROP = {"Surgery", "SOAP / Chart / Progress Notes", "Consult - History and Phy.", "Discharge Summary",
                  "Office Notes", "Letters", "Emergency Room Reports", "General Medicine", "IME-QME-Work Comp etc.",
                  "Pain Management", "Radiology", "Lab Medicine - Pathology", "Autopsy"}

CFG = dict(max_len=DS_CFG["max_len"], batch_size=DS_CFG["batch_size"], epochs=DS_CFG["epochs"], patience=3,
           lr_encoder=3e-5, lr_head=1e-3, weight_decay=0.01, warmup_frac=0.1, dropout=0.2, grad_clip=1.0)

# ---- method constants fixed a priori (not tuned) ----
K_RATIONALE = 10    # PMI terms per class used as RCR supervision
K_EVAL = 10         # PMI terms per class for the (circular, secondary) PMI-RA metric
RHO = 0.20          # rationale size for FR and ERASER@20 (ERASER convention)
FR_MARGIN = 0.20    # comprehensiveness hinge margin
BETA_C_RATIO = 0.5  # beta_c = BETA_C_RATIO * beta_s (one tuned knob instead of two)
HP_F1_TOL = 0.005   # selection rule: within 0.5 F1 points of the best val F1, pick the highest val comprehensiveness
AOPC_BINS = [0.01, 0.05, 0.10, 0.20, 0.50]

# ---- optional inputs ----
AUTO_EXTERNAL_LEXICON = True   # Symptom2Disease: download an independent disease→symptom KB (Kaggle itachi9604)
RATIONALE_LEXICON_CSV = None   # any dataset: clinician/ontology terms, columns `label,term` (overrides the auto lexicon)
HUMAN_RATIONALE_CSV = None     # OPTIONAL (not needed: ade_corpus supplies expert rationales) filled annotation kit: columns `text_id,annotator,word_index` (one row per marked word)
HUMAN_MIN_VOTES = 1            # a word is gold if at least this many annotators marked it
HUMAN_KIT_N = 150              # texts exported for annotation (from the seed-42 test split)
OOD_CSV = None                 # Symptom2Disease only: hand-labelled real dialogues, columns `text,label`

EXTRAS_SEED = 42
VERBOSE = True                        # print progress after every training epoch
LATENCY_N, LATENCY_WARMUP = 100, 10   # GPU / TF-IDF latency samples
LATENCY_CPU_N = 20                    # CPU latency samples for transformer models (main models only)

# ---- output folders (+ resume from a previous Kaggle version attached as input) ----
ON_KAGGLE = os.path.exists("/kaggle")
OUT_ROOT = "/kaggle/working/skatr_revision/base" if ON_KAGGLE else os.path.abspath("./skatr_revision/base")
OUT = os.path.join(OUT_ROOT, DATASET)
os.makedirs(os.path.join(OUT, "preds"), exist_ok=True)
RUNS_FILE = os.path.join(OUT, "runs.jsonl")
EXTRAS_FILE = os.path.join(OUT, "extras.jsonl")
print("Dataset:", DATASET, "| output:", OUT)


Dataset: symptom2disease | output: /kaggle/working/skatr_revision/base/symptom2disease


In [2]:
# SKATR_SETUP 2
# ============================== IMPORTS / ENVIRONMENT ==============================
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score, average_precision_score,
                             cohen_kappa_score, confusion_matrix)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import transformers, sklearn
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, get_linear_schedule_with_warmup

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
ENV = dict(
    python=sys.version.split()[0], torch=torch.__version__, transformers=transformers.__version__,
    sklearn=sklearn.__version__, device=DEVICE,
    gpu=torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    cpu=platform.processor() or platform.machine(), cpu_count=os.cpu_count(),
    cfg=CFG, seeds=SEEDS, main_backbone=MAIN_BACKBONE, backbones=DS_CFG["backbones"], extra_ft=DS_CFG.get("extra_ft", []), llm=LLM_NAME if RUN_LLM else None,
    fixed=dict(K_RATIONALE=K_RATIONALE, K_EVAL=K_EVAL, RHO=RHO, FR_MARGIN=FR_MARGIN, BETA_C_RATIO=BETA_C_RATIO,
               HP_F1_TOL=HP_F1_TOL, AOPC_BINS=AOPC_BINS),
)
json.dump(ENV, open(os.path.join(OUT, "environment.json"), "w"), indent=2)
print(json.dumps(ENV, indent=2))


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


def load_tokenizer(name):
    return AutoTokenizer.from_pretrained(name, use_fast=True)


def load_encoder(name):
    return AutoModel.from_pretrained(name)


{
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "transformers": "5.20.2",
  "sklearn": "1.6.1",
  "device": "cuda",
  "gpu": "Tesla T4",
  "cpu": "x86_64",
  "cpu_count": 4,
  "cfg": {
    "max_len": 128,
    "batch_size": 16,
    "epochs": 8,
    "patience": 3,
    "lr_encoder": 3e-05,
    "lr_head": 0.001,
    "weight_decay": 0.01,
    "warmup_frac": 0.1,
    "dropout": 0.2,
    "grad_clip": 1.0
  },
  "seeds": [
    42,
    123,
    456,
    789,
    2024
  ],
  "main_backbone": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
  "backbones": [
    "distilbert-base-uncased",
    "emilyalsentzer/Bio_ClinicalBERT",
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
  ],
  "extra_ft": [],
  "llm": null,
  "fixed": {
    "K_RATIONALE": 10,
    "K_EVAL": 10,
    "RHO": 0.2,
    "FR_MARGIN": 0.2,
    "BETA_C_RATIO": 0.5,
    "HP_F1_TOL": 0.005,
    "AOPC_BINS": [
      0.01,
      0.05,
      0.1,
      0.2,
      0.5
    ]
  }
}


## 1. Data, independent rationale sources (ADE expert spans, external knowledge base), splits

In [3]:
# SKATR_SETUP 4
# ============================== DATA ==============================
PUBLIC_SOURCES = {
    "symptom2disease": "Kaggle niyarrbarman/symptom2disease (public)",
    "medical_abstracts": "HuggingFace TimSchopf/medical_abstracts (public; Schopf et al., 2022) — conflicting duplicates removed",
    "medical_abstracts_official": "HuggingFace TimSchopf/medical_abstracts (public; Schopf et al., 2022) — as published",
    "mtsamples": "Kaggle tboyle10/medicaltranscriptions (public; scraped from mtsamples.com)",
    "ade_corpus": "HuggingFace ade-benchmark-corpus/ade_corpus_v2 (public; Gurulingappa et al., J Biomed Inform 2012)",
}
ADE_HF = "ade-benchmark-corpus/ade_corpus_v2"


def hf_load(name, config=None):
    try:
        from datasets import load_dataset
    except ImportError:
        os.system("pip install -q datasets")
        from datasets import load_dataset
    return load_dataset(name, config) if config else load_dataset(name)


def kagglehub_dir(handle):
    try:
        import kagglehub
    except ImportError:
        os.system("pip install -q kagglehub")
        import kagglehub
    return kagglehub.dataset_download(handle)


def kaggle_csv(handle, pattern, input_hint=None):
    """Find a CSV of a public Kaggle dataset: attached input first (folder name must contain input_hint), else download."""
    hint = input_hint or handle.split("/")[-1]
    for p in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True):
        if hint.lower() in p.lower():
            return p
    hits = glob.glob(os.path.join(kagglehub_dir(handle), "**", pattern), recursive=True)
    if not hits:
        raise FileNotFoundError(f"{pattern} not found in downloaded {handle}")
    return hits[0]


def raw_dataset():
    """DataFrame with columns label, text and (optionally) split in {'train','test'}."""
    if DATASET == "symptom2disease":
        return pd.read_csv(kaggle_csv("niyarrbarman/symptom2disease", "*ymptom2*isease*.csv", "symptom2"))[["label", "text"]]
    if DATASET == "ade_corpus":
        d = hf_load(ADE_HF, "Ade_corpus_v2_classification")["train"].to_pandas()
        return pd.DataFrame({"label": d["label"].map({0: "Not-Related", 1: "Related"}), "text": d["text"]})
    if DATASET.startswith("medical_abstracts"):
        ds = hf_load("TimSchopf/medical_abstracts")
        names = hf_load("TimSchopf/medical_abstracts", "labels")["train"].to_pandas()
        id2name = dict(zip(names["condition_label"], names["condition_name"]))
        parts = []
        for split in ("train", "test"):
            d = ds[split].to_pandas()
            parts.append(pd.DataFrame({"label": d["condition_label"].map(id2name), "text": d["medical_abstract"], "split": split}))
        return pd.concat(parts, ignore_index=True)
    if DATASET == "mtsamples":
        d = pd.read_csv(kaggle_csv("tboyle10/medicaltranscriptions", "mtsamples*.csv", "medicaltranscriptions"))
        d["label"] = d["medical_specialty"].astype(str).str.strip()
        d = d[~d["label"].isin(MTSAMPLES_DROP)].dropna(subset=["transcription"])
        return d.rename(columns={"transcription": "text"})[["label", "text"]]
    raise ValueError(DATASET)


def load_data():
    df = raw_dataset()
    keep = ["label", "text"] + (["split"] if "split" in df.columns else [])
    df = df[keep].dropna(subset=["label", "text"])
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip()
    df["norm"] = df["text"].str.lower().str.replace(r"[^a-z0-9 ]", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
    n0 = len(df)
    n_label = df.groupby("norm")["label"].nunique()
    conflict_keys = set(n_label[n_label > 1].index)
    n_conflict_rows = int(df["norm"].isin(conflict_keys).sum())
    cross_split = 0
    if DS_CFG["drop_conflicts"]:
        df = df[~df["norm"].isin(conflict_keys)]
        df = df.drop_duplicates("norm")
    else:
        # official benchmark: keep everything except exact duplicates with the SAME label inside the same split
        dcols = ["norm", "label"] + (["split"] if "split" in df.columns else [])
        df = df.drop_duplicates(dcols)
        if "split" in df.columns:
            tr_norm = set(df.loc[df.split == "train", "norm"])
            cross_split = int(df.loc[df.split == "test", "norm"].isin(tr_norm).sum())
    df = df.reset_index(drop=True)
    if DATASET == "mtsamples":
        vc = df["label"].value_counts()
        df = df[df["label"].isin(vc[vc >= MTSAMPLES_MIN_PER_CLASS].index)].reset_index(drop=True)
    info = dict(dataset=DATASET, source=PUBLIC_SOURCES[DATASET], raw_rows=n0,
                rows_in_conflicting_groups=n_conflict_rows, conflicting_groups=len(conflict_keys),
                conflicts_removed=bool(DS_CFG["drop_conflicts"]), final_rows=len(df), n_classes=df["label"].nunique(),
                test_texts_also_in_train=cross_split,
                class_counts=df["label"].value_counts().to_dict(),
                mean_words=float(df["text"].str.split().str.len().mean()))
    return df, info


# ---- tokenisation; negation scope is used only to exclude negated words from gold rationales ----
WORD_RE = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z]+)?|[.,;:!?]")
PUNCT = set(".,;:!?")
NEG_CUES = {"no", "not", "never", "without", "denies", "deny", "denied", "denying", "none", "nor",
            "neither", "cannot", "cant", "dont", "doesnt", "didnt", "havent", "hasnt", "isnt", "negative", "nothing", "absent"}
SCOPE_STOP = {"but", "however", "although", "though", "yet", "except", "besides", "apart"} | PUNCT
MAX_SCOPE = 5
STOP = set(ENGLISH_STOP_WORDS) | {"i", "im", "ive", "ve", "really", "feel", "feeling", "like", "lot", "also", "just", "got", "get", "getting"}


def tokenize_raw(text):
    return WORD_RE.findall(str(text)) or ["."]


def negation_flags(words_lower):
    flags = [False] * len(words_lower)
    for i, w in enumerate(words_lower):
        if w in NEG_CUES or w.endswith("n't"):
            j, n = i + 1, 0
            while j < len(words_lower) and n < MAX_SCOPE and words_lower[j] not in SCOPE_STOP:
                flags[j] = True; j += 1; n += 1
    return flags


def analyse(text):
    raw = tokenize_raw(text)
    low = [w.lower() for w in raw]
    return raw, low, negation_flags(low)


def plain_text(text):
    return " ".join(w for w in analyse(text)[1] if w not in PUNCT)


def mine_rationales(low_list, flag_list, y, n_classes, K, min_count=3):
    """Class-specific PMI terms from the TRAIN split only (used for RCR supervision and the secondary PMI-RA)."""
    N = len(low_list)
    df_w, df_wc = {}, {}
    n_c = np.bincount(y, minlength=n_classes)
    for low, fl, c in zip(low_list, flag_list, y):
        for w in {w for w, f in zip(low, fl) if not f and w.isalpha() and len(w) > 2 and w not in STOP}:
            df_w[w] = df_w.get(w, 0) + 1
            df_wc[(w, c)] = df_wc.get((w, c), 0) + 1
    R = {}
    for c in range(n_classes):
        cands = [(math.log((cnt / n_c[c]) / (df_w[w] / N)), cnt, w) for (w, cc), cnt in df_wc.items() if cc == c and cnt >= min_count]
        cands.sort(key=lambda t: (-t[0], -t[1], t[2]))
        R[c] = set(w for _, _, w in cands[:K])
    return R


def gold_mask(low, flags, label, R):
    S = R.get(label, set())
    return [(w in S) and not f for w, f in zip(low, flags)]


DF, DATA_INFO = load_data()
LABELS = sorted(DF["label"].unique())
L2I = {l: i for i, l in enumerate(LABELS)}
C = len(LABELS)
DF["y"] = DF["label"].map(L2I)
TEXTS = DF["text"].tolist()
Y = DF["y"].values
ANALYSED = [analyse(t) for t in TEXTS]


# ---------------- independent (non-circular) rationale lexicon ----------------
def _norm_name(s):
    return re.sub(r"[^a-z ]", " ", str(s).lower()).replace("  ", " ").strip()


S2D_KB_ALIASES = {  # Symptom2Disease label -> disease name in the itachi9604 knowledge base (spellings differ)
    "peptic ulcer disease": ["peptic ulcer diseae", "peptic ulcer disease"],
    "dimorphic hemorrhoids": ["dimorphic hemmorhoids piles", "dimorphic hemorrhoids piles"],
    "gastroesophageal reflux disease": ["gerd"],
    "chicken pox": ["chicken pox", "chickenpox"],
    "diabetes": ["diabetes"],
    "urinary tract infection": ["urinary tract infection"],
    "common cold": ["common cold"],
}


def external_lexicon_s2d():
    """Disease→symptom KB (Kaggle itachi9604/disease-symptom-description-dataset). Built by its authors, NOT mined from our data."""
    path = kaggle_csv("itachi9604/disease-symptom-description-dataset", "dataset.csv", "disease-symptom")
    kb = pd.read_csv(path)
    scol = [c for c in kb.columns if c.lower().startswith("symptom")]
    kb_terms = {}
    for _, r in kb.iterrows():
        dn = _norm_name(r["Disease"])
        for c in scol:
            if isinstance(r[c], str):
                for w in re.split(r"[_\s]+", r[c].strip().lower()):
                    if w.isalpha() and len(w) > 2 and w not in STOP:
                        kb_terms.setdefault(dn, set()).add(w)
    names = list(kb_terms)
    lex, mapping = {}, {}
    for lab in LABELS:
        ln = _norm_name(lab)
        cands = S2D_KB_ALIASES.get(ln, []) + [ln]
        hit = next((c for c in cands if c in kb_terms), None)
        if hit is None:
            close = difflib.get_close_matches(ln, names, n=1, cutoff=0.8)
            hit = close[0] if close else None
        mapping[lab] = hit
        if hit:
            lex[L2I[lab]] = set(kb_terms[hit])
    # drop words that appear for more than half of the mapped classes (e.g. "high", "pain"): not class-specific
    cnt = pd.Series([w for s in lex.values() for w in s]).value_counts()
    generic = set(cnt[cnt > 0.5 * max(len(lex), 1)].index)
    lex = {c: s - generic for c, s in lex.items()}
    return lex, mapping, sorted(generic)


EXT_LEX, EXT_LEX_SOURCE = None, None
if RATIONALE_LEXICON_CSV:
    lex_df = pd.read_csv(RATIONALE_LEXICON_CSV)
    EXT_LEX = {L2I[l]: set(g["term"].astype(str).str.lower()) for l, g in lex_df.groupby("label") if l in L2I}
    EXT_LEX_SOURCE = f"user file {RATIONALE_LEXICON_CSV}"
elif AUTO_EXTERNAL_LEXICON and DATASET == "symptom2disease":
    try:
        EXT_LEX, _map, _gen = external_lexicon_s2d()
        EXT_LEX_SOURCE = "Kaggle itachi9604/disease-symptom-description-dataset"
        json.dump(dict(source=EXT_LEX_SOURCE, label_to_kb_disease=_map, generic_words_removed=_gen,
                       terms={LABELS[c]: sorted(s) for c, s in EXT_LEX.items()}),
                  open(os.path.join(OUT, "external_lexicon.json"), "w"), indent=2)
        print(f"External lexicon: {len(EXT_LEX)}/{C} classes mapped; unmapped:", [l for l, v in _map.items() if v is None])
    except Exception as e:  # never let an optional resource kill the run
        print("External lexicon unavailable:", repr(e)); EXT_LEX = None
if EXT_LEX:
    pmi_all = mine_rationales([a[1] for a in ANALYSED], [a[2] for a in ANALYSED], Y, C, K_EVAL)
    ov = [len(EXT_LEX[c] & pmi_all[c]) / max(len(pmi_all[c]), 1) for c in EXT_LEX]
    DATA_INFO["external_lexicon"] = dict(source=EXT_LEX_SOURCE, classes=len(EXT_LEX),
                                         mean_terms=float(np.mean([len(s) for s in EXT_LEX.values()])),
                                         overlap_with_pmi_top10=float(np.mean(ov)))

# ---------------- human rationales ----------------
# (a) ADE-Corpus-V2: drug + adverse-effect spans marked by the corpus's trained annotators (Gurulingappa et al., 2012)
#     are used as gold rationales for 'Related' sentences: no new annotation needed.
# (b) any dataset: a filled annotation kit (HUMAN_RATIONALE_CSV), optional.
HUMAN, HUMAN_SOURCE = None, None   # HUMAN: dict text_id -> set(word_index)


def ade_expert_rationales():
    """Map the expert character spans of the relation config onto our word tokens (union over all relations of a sentence)."""
    rel = hf_load(ADE_HF, "Ade_corpus_v2_drug_ade_relation")["train"]
    by_text = {}
    n_rel = 0
    for r in rel:
        raw = str(r["text"])
        spans = []
        for ent in ("drug", "effect"):
            ix = r["indexes"][ent]
            spans += list(zip(list(ix["start_char"]), list(ix["end_char"])))
        words = [(m.start(), m.end()) for m in WORD_RE.finditer(raw)]
        marked = {j for j, (a, b) in enumerate(words) if any(a < e and b > s for s, e in spans)}
        by_text.setdefault(raw.strip(), set()).update(marked); n_rel += 1
    t2id = {t: i for i, t in enumerate(TEXTS)}
    gold = {}
    for t, s in by_text.items():
        i = t2id.get(t)
        if i is not None and LABELS[Y[i]] == "Related" and s:
            gold[i] = s
    return gold, n_rel, len(by_text)


if DATASET == "ade_corpus" and not HUMAN_RATIONALE_CSV:
    HUMAN, _n_rel, _n_txt = ade_expert_rationales()
    HUMAN_SOURCE = "expert"
    n_pos = int(sum(LABELS[y] == "Related" for y in Y))
    frac = [len(HUMAN[i]) / max(1, sum(w not in PUNCT for w in ANALYSED[i][1])) for i in HUMAN]
    DATA_INFO["human_rationales"] = dict(
        source="ADE-Corpus-V2 expert drug/effect spans (Gurulingappa et al., 2012)", relations=_n_rel,
        sentences_with_spans=_n_txt, related_sentences_with_gold=len(HUMAN), related_sentences=n_pos,
        mean_fraction_of_words_marked=float(np.mean(frac)) if frac else None,
        note="Inter-annotator agreement: cite the value reported in the original corpus paper.")
    print("Expert rationales:", DATA_INFO["human_rationales"])

if HUMAN_RATIONALE_CSV:
    HUMAN_SOURCE = "manual"
    hr = pd.read_csv(HUMAN_RATIONALE_CSV)
    anns = sorted(hr["annotator"].unique())
    votes = hr.groupby(["text_id", "word_index"])["annotator"].nunique()
    HUMAN = {}
    for (tid, wi), v in votes.items():
        if v >= HUMAN_MIN_VOTES:
            HUMAN.setdefault(int(tid), set()).add(int(wi))
    ids = sorted(hr["text_id"].unique())
    for tid in ids:
        HUMAN.setdefault(int(tid), set())
    kappa = None
    if len(anns) >= 2:
        a1, a2 = [], []
        for tid in ids:
            n = len(ANALYSED[int(tid)][1])
            s1 = set(hr[(hr.text_id == tid) & (hr.annotator == anns[0])].word_index)
            s2 = set(hr[(hr.text_id == tid) & (hr.annotator == anns[1])].word_index)
            for j, w in enumerate(ANALYSED[int(tid)][1]):
                if w not in PUNCT:
                    a1.append(j in s1); a2.append(j in s2)
        kappa = float(cohen_kappa_score(a1, a2))
    DATA_INFO["human_rationales"] = dict(texts=len(ids), annotators=len(anns), token_level_cohen_kappa=kappa,
                                         min_votes=HUMAN_MIN_VOTES)
    print("Human rationales:", DATA_INFO["human_rationales"])

OOD = None
if OOD_CSV and DATASET == "symptom2disease":
    ood_df = pd.read_csv(OOD_CSV)
    ood_df = ood_df[ood_df["label"].isin(L2I)].reset_index(drop=True)
    OOD = dict(texts=ood_df["text"].astype(str).tolist(), y=ood_df["label"].map(L2I).values)

print({k: v for k, v in DATA_INFO.items() if k != "class_counts"})
json.dump(DATA_INFO, open(os.path.join(OUT, "data_info.json"), "w"), indent=2, default=str)


def make_split(seed):
    idx = np.arange(len(DF))
    if DS_CFG["fixed_test"]:
        te = idx[DF["split"].values == "test"]
        trv = idx[DF["split"].values == "train"]
        tr, va = train_test_split(trv, test_size=0.15, stratify=Y[trv], random_state=seed)
        return tr, va, te
    tr, tmp = train_test_split(idx, test_size=0.30, stratify=Y, random_state=seed)
    va, te = train_test_split(tmp, test_size=0.50, stratify=Y[tmp], random_state=seed)
    return tr, va, te


External lexicon: 24/24 classes mapped; unmapped: []
{'dataset': 'symptom2disease', 'source': 'Kaggle niyarrbarman/symptom2disease (public)', 'raw_rows': 1200, 'rows_in_conflicting_groups': 0, 'conflicting_groups': 0, 'conflicts_removed': True, 'final_rows': 1153, 'n_classes': 24, 'test_texts_also_in_train': 0, 'mean_words': 30.62532523850824, 'external_lexicon': {'source': 'Kaggle itachi9604/disease-symptom-description-dataset', 'classes': 24, 'mean_terms': 11.125, 'overlap_with_pmi_top10': 0.18333333333333335}}


In [4]:
# SKATR_SETUP 5
# ============================== LEXICAL MODELS + PER-SEED DATA ==============================
class LexicalModel:
    """TF-IDF (1-2 grams) + LogReg / LinearSVM. Native explanation = exact linear contribution coef × tfidf of each unigram."""

    def __init__(self, clf, C_reg):
        self.clf_name, self.C_reg = clf, C_reg

    def fit(self, texts, y):
        self.vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, lowercase=False, token_pattern=r"(?u)\b[\w']+\b")
        X = self.vec.fit_transform([plain_text(t) for t in texts])
        self.clf = LogisticRegression(C=self.C_reg, max_iter=5000) if self.clf_name == "lr" else LinearSVC(C=self.C_reg, max_iter=20000)
        self.clf.fit(X, y)
        self.vocab = self.vec.vocabulary_
        return self

    def _X(self, texts):
        return self.vec.transform([plain_text(t) for t in texts])

    def predict(self, texts):
        return self.clf.predict(self._X(texts))

    def predict_proba(self, texts):
        X = self._X(texts)
        if self.clf_name == "lr":
            return self.clf.predict_proba(X)
        d = self.clf.decision_function(X)       # SVM: softmax of margins (uncalibrated; stated in the paper)
        if d.ndim == 1:                          # binary task: sklearn returns one margin for class 1
            d = np.stack([-d / 2, d / 2], 1)
        e = np.exp(d - d.max(1, keepdims=True))
        return e / e.sum(1, keepdims=True)

    def predict_logp(self, texts):
        return np.log(np.clip(self.predict_proba(texts), 1e-9, 1.0))

    def _coef(self, c):
        W = self.clf.coef_
        if W.shape[0] == 1:                      # binary task: one weight row = evidence for class 1
            return W[0] if c == 1 else -W[0]
        return W[c]

    def native_scores(self, texts, classes):
        X = self._X(texts).tocsr()
        out = []
        for i, (t, c) in enumerate(zip(texts, classes)):
            row = X.getrow(i); val = dict(zip(row.indices, row.data))
            coef = self._coef(int(c))
            s = []
            for w in analyse(t)[1]:
                j = self.vocab.get(w)
                s.append(-1e9 if w in PUNCT else (float(coef[j] * val.get(j, 0.0)) if j is not None else 0.0))
            out.append(np.array(s))
        return out


def tune_lexical(clf, tr_texts, ytr, va_texts, yva):
    grid = [0.1, 1, 10, 100] if clf == "lr" else [0.01, 0.1, 1, 10]
    best, best_f1 = None, -1
    for c in grid:
        m = LexicalModel(clf, c).fit(tr_texts, ytr)
        f = f1_score(yva, m.predict(va_texts), average="macro")
        if f > best_f1:
            best, best_f1 = m, f
    return best, best_f1


def oof_logp(C_reg, tr_texts, ytr, seed):
    """Out-of-fold lexical log-probs for the TRAIN split (stacking without leakage)."""
    out = np.zeros((len(tr_texts), C))
    tr_texts = np.array(tr_texts, dtype=object)
    for a, b in StratifiedKFold(n_splits=5, shuffle=True, random_state=seed).split(tr_texts, ytr):
        out[b] = LexicalModel("lr", C_reg).fit(list(tr_texts[a]), ytr[a]).predict_logp(list(tr_texts[b]))
    return out


_SEED_CACHE = {}


def seed_data(seed):
    if seed in _SEED_CACHE:
        return _SEED_CACHE[seed]
    tr, va, te = make_split(seed)
    T = lambda idx: [TEXTS[i] for i in idx]
    d = dict(seed=seed, tr=tr, va=va, te=te, texts=dict(tr=T(tr), va=T(va), te=T(te)), y=dict(tr=Y[tr], va=Y[va], te=Y[te]))
    lows = [ANALYSED[i][1] for i in tr]; fls = [ANALYSED[i][2] for i in tr]
    d["R"] = {K: mine_rationales(lows, fls, Y[tr], C, K) for K in sorted({K_RATIONALE, K_EVAL})}
    d["lex"], d["lex_val_f1"] = {}, {}
    for clf in ("lr", "svm"):
        d["lex"][clf], d["lex_val_f1"][clf] = tune_lexical(clf, d["texts"]["tr"], d["y"]["tr"], d["texts"]["va"], d["y"]["va"])
    d["oof"] = oof_logp(d["lex"]["lr"].C_reg, d["texts"]["tr"], d["y"]["tr"], seed)
    # fixed explanation subset (same texts for every model of this seed -> paired comparisons)
    rng = np.random.RandomState(seed)
    if DS_CFG.get("explain_pos_neg") and HUMAN:
        # ADE: n_pos sentences that carry expert rationales + n_neg 'Not-Related' sentences (faithfulness on both classes)
        n_pos, n_neg = DS_CFG["explain_pos_neg"]
        has = [j for j, gi in enumerate(te) if int(gi) in HUMAN]
        hs = set(has)
        rest = [j for j in range(len(te)) if j not in hs]
        pos = rng.permutation(has)[:n_pos].tolist() + rng.permutation(rest)[:n_neg].tolist()
    else:
        pos = rng.permutation(len(te))[: DS_CFG["n_explain"]].tolist()
        if HUMAN and HUMAN_SOURCE == "manual" and DS_CFG["n_explain"] > 0:
            pos += [j for j, gi in enumerate(te) if int(gi) in HUMAN and j not in pos]
    d["explain_pos"] = np.array(sorted(pos), dtype=int)
    _SEED_CACHE[seed] = d
    return d


## 2. Models: SKAT-R (RCR + faithfulness regulariser) and zero-shot LLM baseline

In [5]:
# SKATR_SETUP 7
# ============================== NEURAL MODEL: SKAT-R v3 ==============================
class SKATR(nn.Module):
    """
    logits = s · lex_logp(x) + ctx(x)      (lexical prior as a residual; no gate — the v2 gate had no measurable effect)
    ctx(x) = W · Σ_t a_t h_t               (attention pooling over word-piece states, CLS/SEP/punctuation excluded)
    Training loss = CE + α·JS(a, PMI-rationale)                         (RCR: plausibility)
                       + β_s·KL(p_ctx(x) || p_ctx(x restricted to top-ρ words))        (FR sufficiency)
                       + β_c·max(0, p_ctx(y | x without top-ρ words) − p_ctx(y | x) + m) (FR comprehensiveness)
    pooling='cls', use_lex=False, α=β=0  ==> standard fine-tuning baseline.
    """

    def __init__(self, enc_name, n_cls, pooling="attn", use_lex=True, dropout=0.2):
        super().__init__()
        self.encoder = load_encoder(enc_name)
        H = self.encoder.config.hidden_size
        self.pooling, self.use_lex = pooling, use_lex
        self.attn_scorer = nn.Sequential(nn.Linear(H, 128), nn.Tanh(), nn.Linear(128, 1))
        self.dropout = nn.Dropout(dropout)
        self.ctx_head = nn.Linear(H, n_cls)
        if use_lex:
            self.lex_scale = nn.Parameter(torch.tensor(1.0))

    def head_parameters(self):
        return [p for n, p in self.named_parameters() if not n.startswith("encoder.")]

    def forward(self, input_ids=None, attention_mask=None, valid=None, lex=None, inputs_embeds=None):
        if inputs_embeds is not None:
            hs = self.encoder(inputs_embeds=inputs_embeds, attention_mask=attention_mask).last_hidden_state
        else:
            hs = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        a = None
        if self.pooling == "attn":
            s = self.attn_scorer(hs).squeeze(-1).float().masked_fill(valid == 0, -1e4)
            a = torch.softmax(s, dim=-1)
            pooled = torch.bmm(a.unsqueeze(1).to(hs.dtype), hs).squeeze(1)
        else:
            pooled = hs[:, 0]
        ctx = self.ctx_head(self.dropout(pooled)).float()
        logits = self.lex_scale * lex.float() + ctx if self.use_lex else ctx
        return logits, a, ctx


def js_divergence(p, q, eps=1e-8):
    m = 0.5 * (p + q)
    kl = lambda x, y: (x * (torch.log(x + eps) - torch.log(y + eps))).sum(-1)
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


def featurize(tok, texts, lex_logp, labels=None, R=None, max_len=128):
    an = [analyse(t) for t in texts]
    enc = tok([a[0] for a in an], is_split_into_words=True, truncation=True, max_length=max_len, padding="max_length")
    n = len(texts)
    valid = np.zeros((n, max_len), dtype=np.float32)
    rmask = np.zeros((n, max_len), dtype=np.float32)
    widx = -np.ones((n, max_len), dtype=np.int64)
    for i in range(n):
        gm = gold_mask(an[i][1], an[i][2], int(labels[i]), R) if (labels is not None and R is not None) else None
        for j, w in enumerate(enc.word_ids(i)):
            if w is None:
                continue
            widx[i, j] = w
            if an[i][1][w] not in PUNCT:
                valid[i, j] = 1.0
            if gm is not None and gm[w]:
                rmask[i, j] = 1.0
    return dict(input_ids=torch.tensor(enc["input_ids"]), attention_mask=torch.tensor(enc["attention_mask"]),
                valid=torch.tensor(valid), rmask=torch.tensor(rmask), widx=torch.tensor(widx),
                lex=torch.tensor(lex_logp, dtype=torch.float32), n_words=[len(a[0]) for a in an])


SEQ_KEYS = ("input_ids", "attention_mask", "valid", "rmask", "widx")


def trim(f, rows=None):
    """Dynamic padding: cut right-padded sequence tensors to the longest real sequence of these rows.
    Numerically equivalent (padding is masked everywhere), but much faster for short texts."""
    sel = (lambda v: v[rows]) if rows is not None else (lambda v: v)
    L = max(1, int(sel(f["attention_mask"]).sum(1).max()))
    return {k: (sel(f[k])[:, :L] if k in SEQ_KEYS else sel(f[k])) for k in f if k != "n_words"}


def fr_keep_mask(a, widx, valid, rho):
    """Whole-word mask of the top-ρ fraction of content words ranked by attention (detached)."""
    keep = torch.zeros_like(valid)
    for i in range(a.shape[0]):
        vi = valid[i] > 0
        if not bool(vi.any()):
            continue
        w = widx[i][vi]
        ws = torch.zeros(int(w.max()) + 1, device=a.device).scatter_add_(0, w, a[i][vi])
        present = torch.zeros_like(ws, dtype=torch.bool); present[w] = True
        k = max(1, int(math.ceil(rho * int(present.sum()))))
        ws = ws.masked_fill(~present, -1.0)
        top = torch.topk(ws, k).indices
        keep[i] = (torch.isin(widx[i], top) & vi).float()
    return keep


class NeuralPipeline:
    def __init__(self, model, tok, lex_model, max_len):
        self.model, self.tok, self.lex, self.max_len = model, tok, lex_model, max_len

    def feats(self, texts):
        lex = self.lex.predict_logp(texts) if self.lex is not None else np.zeros((len(texts), C))
        return featurize(self.tok, texts, lex, max_len=self.max_len)

    def _fwd(self, f, sl=slice(None)):
        t = trim(f)
        b = {k: t[k][sl].to(DEVICE) for k in ("input_ids", "attention_mask", "valid", "lex")}
        with torch.autocast(device_type="cuda", enabled=USE_AMP and DEVICE == "cuda"):
            return self.model(**b)

    @torch.no_grad()
    def predict_proba(self, texts, bs=64):
        """Texts are processed sorted by length (with dynamic padding), then returned in the original order."""
        self.model.eval()
        texts = list(texts)
        if not texts:
            return np.zeros((0, C))
        order = np.argsort([len(t) for t in texts], kind="stable")
        out = np.zeros((len(texts), C), dtype=np.float32)
        for s in range(0, len(texts), bs):
            ix = order[s:s + bs]
            logits, _, _ = self._fwd(self.feats([texts[i] for i in ix]))
            out[ix] = torch.softmax(logits.float(), -1).cpu().numpy()
        return out

    @torch.no_grad()
    def attention_word_scores(self, texts, bs=64):
        self.model.eval()
        res = []
        for s in range(0, len(texts), bs):
            f = self.feats(texts[s:s + bs])
            _, a, _ = self._fwd(f)
            a = a.float().cpu().numpy()
            widx = trim(f)["widx"].numpy()
            for i in range(a.shape[0]):
                res.append(_word_agg(a[i], widx[i], f["n_words"][i]))
        return res

    def grad_word_scores(self, texts):
        bs = max(4, 32 * 128 // self.max_len)
        self.model.eval()
        emb = self.model.encoder.get_input_embeddings()
        res = []
        for s in range(0, len(texts), bs):
            f0 = self.feats(texts[s:s + bs])
            f = trim(f0)
            e = emb(f["input_ids"].to(DEVICE)).detach().requires_grad_(True)
            logits, _, _ = self.model(inputs_embeds=e, attention_mask=f["attention_mask"].to(DEVICE),
                                      valid=f["valid"].to(DEVICE), lex=f["lex"].to(DEVICE))
            g = torch.autograd.grad(logits.gather(1, logits.argmax(1, keepdim=True)).sum(), e)[0]
            tok_scores = ((g * e).sum(-1).abs() * f["valid"].to(DEVICE)).detach().cpu().numpy()
            for i in range(tok_scores.shape[0]):
                res.append(_word_agg(tok_scores[i], f["widx"][i].numpy(), f0["n_words"][i]))
        return res

    def native_scores(self, texts, classes=None):
        return self.attention_word_scores(texts) if self.model.pooling == "attn" else self.grad_word_scores(texts)


def _word_agg(tok_scores, widx, n_words):
    m = widx >= 0
    return np.bincount(widx[m], weights=tok_scores[m], minlength=n_words)[:n_words]


_TOK_CACHE = {}


def get_tok(name):
    if name not in _TOK_CACHE:
        _TOK_CACHE[name] = load_tokenizer(name)
    return _TOK_CACHE[name]


def train_neural(spec, d, seed, keep_log=False):
    set_seed(seed)
    tok = get_tok(spec["enc"])
    lex_model = d["lex"]["lr"] if spec["use_lex"] else None
    R = d["R"][spec.get("K", K_RATIONALE)]
    tr_lex = d["oof"] if spec["use_lex"] else np.zeros((len(d["tr"]), C))
    ftr = featurize(tok, d["texts"]["tr"], tr_lex, d["y"]["tr"], R, CFG["max_len"])
    ytr = torch.tensor(d["y"]["tr"])
    model = SKATR(spec["enc"], C, spec["pooling"], spec["use_lex"], CFG["dropout"]).to(DEVICE)
    pipe = NeuralPipeline(model, tok, lex_model, CFG["max_len"])
    opt = torch.optim.AdamW([{"params": list(model.encoder.parameters()), "lr": CFG["lr_encoder"]},
                             {"params": model.head_parameters(), "lr": CFG["lr_head"]}], weight_decay=CFG["weight_decay"])
    n, bs = len(ytr), CFG["batch_size"]
    steps = CFG["epochs"] * math.ceil(n / bs)
    sched = get_linear_schedule_with_warmup(opt, int(CFG["warmup_frac"] * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    alpha, beta_s, beta_c = spec.get("alpha", 0.0), spec.get("beta_s", 0.0), spec.get("beta_c", 0.0)
    use_fr = spec["pooling"] == "attn" and (beta_s > 0 or beta_c > 0)
    gen = torch.Generator().manual_seed(seed)
    best_f1, best_state, bad, epochs_run = -1.0, None, 0, 0
    t0 = time.time()
    for ep in range(CFG["epochs"]):
        model.train(); epochs_run += 1
        perm = torch.randperm(n, generator=gen)
        for s in range(0, n, bs):
            ix = perm[s:s + bs]
            fb = trim(ftr, ix)                                   # dynamic padding for this batch
            b = {k: fb[k].to(DEVICE) for k in ("input_ids", "attention_mask", "valid", "lex")}
            yb = ytr[ix].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", enabled=USE_AMP):
                logits, a, ctx = model(**b)
            loss = F.cross_entropy(logits.float(), yb)
            if alpha > 0 and a is not None:
                r = fb["rmask"].to(DEVICE)
                has = r.sum(-1) > 0
                if has.any():
                    loss = loss + alpha * js_divergence(a.float()[has], r[has] / r[has].sum(-1, keepdim=True)).mean()
            scaler.scale(loss).backward()
            if use_fr:
                # FR: separate forward/backward passes (targets are detached) -> memory of one forward at a time
                widx = fb["widx"].to(DEVICE)
                keep = fr_keep_mask(a.detach().float(), widx, b["valid"], RHO)
                special = ((widx < 0).float() * b["attention_mask"].float())
                p_full = torch.softmax(ctx.detach().float(), -1)
                if beta_s > 0:
                    am_s = torch.clamp(special + keep, max=1.0).long()
                    with torch.autocast(device_type="cuda", enabled=USE_AMP):
                        _, _, ctx_s = model(input_ids=b["input_ids"], attention_mask=am_s, valid=b["valid"] * keep, lex=b["lex"])
                    l_s = (p_full * (torch.log(p_full + 1e-8) - F.log_softmax(ctx_s.float(), -1))).sum(-1).mean()
                    scaler.scale(beta_s * l_s).backward()
                if beta_c > 0:
                    v_c = b["valid"] * (1 - keep)
                    ok = v_c.sum(-1) > 0
                    if ok.any():
                        am_c = (b["attention_mask"].float() * (1 - keep)).long()
                        with torch.autocast(device_type="cuda", enabled=USE_AMP):
                            _, _, ctx_c = model(input_ids=b["input_ids"][ok], attention_mask=am_c[ok], valid=v_c[ok], lex=b["lex"][ok])
                        ar = torch.arange(int(ok.sum()), device=DEVICE)
                        p_c = torch.softmax(ctx_c.float(), -1)[ar, yb[ok]]
                        l_c = F.relu(p_c - p_full[ok][ar, yb[ok]] + FR_MARGIN).mean()
                        scaler.scale(beta_c * l_c).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            scaler.step(opt); scaler.update(); sched.step()
        vf1 = f1_score(d["y"]["va"], pipe.predict_proba(d["texts"]["va"]).argmax(1), average="macro")
        if VERBOSE:
            print(f"    epoch {ep+1}/{CFG['epochs']}: val macro-F1={100*vf1:.2f}  ({time.time()-t0:.0f}s elapsed)", flush=True)
        if vf1 > best_f1 + 1e-6:
            best_f1, bad = vf1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= CFG["patience"]:
                break
    model.load_state_dict(best_state)
    n_params = sum(p.numel() for p in model.parameters())
    return pipe, dict(val_macro_f1=best_f1, epochs_run=epochs_run, train_time_s=time.time() - t0, n_params=int(n_params))


# ============================== ZERO-SHOT LLM BASELINE ==============================
class ZeroShotLLM:
    """Lists the classes as lettered options and reads the next-token distribution over the letters.
    One forward pass per text, deterministic, and it yields class probabilities (so calibration can be reported)."""

    def __init__(self, name, labels, task, max_text_tokens):
        self.tok = AutoTokenizer.from_pretrained(name)
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        dt = torch.float16 if DEVICE == "cuda" else torch.float32
        try:
            self.model = AutoModelForCausalLM.from_pretrained(name, dtype=dt)
        except TypeError:
            self.model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dt)
        self.model.to(DEVICE).eval()
        assert len(labels) <= 26, "letter scoring supports up to 26 classes"
        self.labels, self.task, self.max_text_tokens = labels, task, max_text_tokens
        self.letters = [chr(65 + i) for i in range(len(labels))]
        self.letter_ids = [self.tok.encode(L, add_special_tokens=False)[0] for L in self.letters]
        assert len(set(self.letter_ids)) == len(self.letters)

    def prompt(self, text):
        ids = self.tok.encode(str(text), add_special_tokens=False)[: self.max_text_tokens]
        opts = "\n".join(f"{L}. {lab}" for L, lab in zip(self.letters, self.labels))
        msgs = [{"role": "system", "content": "You are a careful clinical text classification assistant."},
                {"role": "user", "content": f"{self.task}\n\nCategories:\n{opts}\n\nText:\n\"\"\"{self.tok.decode(ids)}\"\"\"\n\n"
                                            "Answer with the single letter of the best category."}]
        return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    @torch.no_grad()
    def predict_proba(self, texts, bs=None):
        bs = bs or (8 if self.max_text_tokens <= 512 else 4)
        out = []
        for s in range(0, len(texts), bs):
            enc = self.tok([self.prompt(t) for t in texts[s:s + bs]], return_tensors="pt", padding=True,
                           add_special_tokens=False).to(DEVICE)
            try:
                logits = self.model(**enc, logits_to_keep=1).logits   # only the last position: saves GPU memory
            except TypeError:
                logits = self.model(**enc).logits
            lg = logits[:, -1, :][:, self.letter_ids].float()
            out.append(torch.softmax(lg, -1).cpu().numpy())
        return np.concatenate(out)


_LLM_CACHE = {}


def llm_probs():
    """Probabilities for every text that is a test text in any seed; cached to disk (inference only, no training)."""
    if _LLM_CACHE:
        return _LLM_CACHE
    path = os.path.join(OUT, "llm_probs.npz")
    need = sorted({int(i) for s in SEEDS for i in make_split(s)[2]})
    if os.path.exists(path):
        z = np.load(path)
        _LLM_CACHE.update({int(i): p for i, p in zip(z["idx"], z["probs"])})
    todo = [i for i in need if i not in _LLM_CACHE]
    if todo:
        llm = ZeroShotLLM(LLM_NAME, LABELS, DS_CFG["task"], DS_CFG["llm_max_tokens"])
        t0 = time.time()
        P = llm.predict_proba([TEXTS[i] for i in todo])
        print(f"LLM scored {len(todo)} texts in {time.time()-t0:.0f}s")
        _LLM_CACHE.update({i: p for i, p in zip(todo, P)})
        idx = np.array(sorted(_LLM_CACHE)); np.savez(path, idx=idx, probs=np.stack([_LLM_CACHE[i] for i in idx]))
        del llm
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return _LLM_CACHE


## 3. Metrics: accuracy, calibration, plausibility (one method for all models), ERASER/AOPC faithfulness

In [6]:
# SKATR_SETUP 9
# ============================== METRICS: accuracy, calibration, plausibility, faithfulness ==============================
def cls_metrics(y, P, prefix="test_"):
    p = P.argmax(1)
    out = {prefix + "acc": accuracy_score(y, p), prefix + "macro_f1": f1_score(y, p, average="macro"),
           prefix + "weighted_f1": f1_score(y, p, average="weighted"),
           prefix + "macro_p": precision_score(y, p, average="macro", zero_division=0),
           prefix + "macro_r": recall_score(y, p, average="macro", zero_division=0)}
    out.update(calibration(y, P, prefix))
    return out


def calibration(y, P, prefix="test_", n_bins=15):
    conf, pred = P.max(1), P.argmax(1)
    corr = (pred == y).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(corr[m].mean() - conf[m].mean())
    onehot = np.eye(P.shape[1])[y]
    return {prefix + "ece": float(ece), prefix + "brier": float(((P - onehot) ** 2).sum(1).mean()),
            prefix + "nll": float(-np.log(np.clip(P[np.arange(len(y)), y], 1e-12, 1)).mean())}


def _content(low):
    return np.array([w not in PUNCT for w in low])


def plausibility(texts, labels, word_scores, gold_fn):
    """gold_fn(i, low, flags, y) -> bool mask. Returns R-precision, token AUPRC and their random-ranking expectations."""
    rp, rp_rand, ap, ap_rand = [], [], [], []
    for i, (t, y, s) in enumerate(zip(texts, labels, word_scores)):
        _, low, fl = analyse(t)
        g = np.array(gold_fn(i, low, fl, int(y)), bool)
        cm = _content(low)
        g = g & cm
        k = int(g.sum())
        if k == 0:
            continue
        s = np.array(s, float)[: len(g)]
        s = np.concatenate([s, np.zeros(len(g) - len(s))]) if len(s) < len(g) else s
        s = np.where(cm, s, -1e9)
        top = np.argsort(-s, kind="stable")[:k]
        rp.append(g[top].mean()); rp_rand.append(k / cm.sum())
        if k < cm.sum():
            ap.append(average_precision_score(g[cm], s[cm])); ap_rand.append(k / cm.sum())
    f = lambda v: float(np.mean(v)) if v else float("nan")
    return dict(rp=f(rp), rp_random=f(rp_rand), auprc=f(ap), auprc_random=f(ap_rand), n=len(rp))


def lexicon_gold(R):
    return lambda i, low, fl, y: gold_mask(low, fl, y, R)


def occlusion_scores(proba_fn, texts, max_words, chunk=2048):
    """Model-agnostic leave-one-word-out: score_i = p(ŷ|x) − p(ŷ|x without word i). Same method for every model."""
    P = proba_fn(list(texts)); yhat = P.argmax(1)
    variants, owner = [], []
    plan = []
    for n, t in enumerate(texts):
        raw, low, _ = analyse(t)
        cidx = [i for i, w in enumerate(low) if w not in PUNCT][:max_words]
        plan.append((len(raw), cidx))
        for i in cidx:
            variants.append(" ".join(raw[:i] + raw[i + 1:]) or "."); owner.append(n)
    probs = np.concatenate([proba_fn(variants[s:s + chunk]) for s in range(0, len(variants), chunk)]) if variants else np.zeros((0, C))
    out, k = [], 0
    for n, (nraw, cidx) in enumerate(plan):
        sc = np.zeros(nraw)
        for i in cidx:
            sc[i] = P[n, yhat[n]] - probs[k, yhat[n]]; k += 1
        out.append(sc)
    return out


def eraser_aopc(proba_fn, texts, word_scores, seed, bins=AOPC_BINS, frac=RHO):
    """ERASER comprehensiveness/sufficiency at `frac` and AOPC over `bins`, with a random-rationale control."""
    rng = np.random.RandomState(seed)
    jobs = []   # (text_id, bin, kind, text)
    for n, (t, s) in enumerate(zip(texts, word_scores)):
        raw, low, _ = analyse(t)
        content = [i for i, w in enumerate(low) if w not in PUNCT]
        if not content:
            continue
        s = np.array(s, float)
        sc = np.array([s[i] if i < len(s) else 0.0 for i in content])
        order = np.array(content)[np.argsort(-sc, kind="stable")]
        rorder = rng.permutation(content)
        for b in sorted(set(bins) | {frac}):
            k = max(1, int(math.ceil(b * len(content))))
            for tag, ordr in (("m", order), ("r", rorder)):
                top = set(ordr[:k].tolist())
                jobs.append((n, b, tag + "_minus", " ".join(raw[i] for i in range(len(raw)) if i not in top) or "."))
                jobs.append((n, b, tag + "_only", " ".join(raw[i] for i in range(len(raw)) if i in top) or "."))
    P = proba_fn(list(texts)); yhat = P.argmax(1)
    Q = proba_fn([j[3] for j in jobs]) if jobs else np.zeros((0, C))
    rows = pd.DataFrame([dict(n=j[0], b=j[1], kind=j[2], drop=P[j[0], yhat[j[0]]] - Q[q, yhat[j[0]]]) for q, j in enumerate(jobs)])
    if rows.empty:
        return {}
    g = rows.groupby(["b", "kind"])["drop"].mean()
    out = dict(comp20=g[(frac, "m_minus")], suff20=g[(frac, "m_only")],
               comp20_random=g[(frac, "r_minus")], suff20_random=g[(frac, "r_only")],
               aopc_comp=float(np.mean([g[(b, "m_minus")] for b in bins])), aopc_suff=float(np.mean([g[(b, "m_only")] for b in bins])),
               aopc_comp_random=float(np.mean([g[(b, "r_minus")] for b in bins])),
               aopc_suff_random=float(np.mean([g[(b, "r_only")] for b in bins])))
    return {k: float(v) for k, v in out.items()}


# ---- synthetic robustness probes (Symptom2Disease only; reported as stress tests, not clinical validation) ----
NEG_TEMPLATES = ["I do not have any {}.", "There is no {}.", "I don't have {}.", "No {} at all."]
OPENERS = ["Hello doctor, I hope you are doing well. ", "Hi, I need some advice please. ", "Good evening doctor. "]
CLOSERS = [" What could this be?", " Please tell me what I should do.", " Thanks in advance for your help."]


def make_probes(texts, labels, R, seed):
    rng = np.random.RandomState(seed)
    neg_texts, neg_idx, distract = [], [], []
    for i, (t, y) in enumerate(zip(texts, labels)):
        present = set(analyse(t)[1])
        others = [c for c in range(C) if c != y]
        rng.shuffle(others)
        for c in others:
            terms = sorted(w for w in R[c] if w not in present and w not in R[int(y)])
            if terms:
                neg_texts.append(t.rstrip() + " " + NEG_TEMPLATES[rng.randint(len(NEG_TEMPLATES))].format(terms[rng.randint(len(terms))]))
                neg_idx.append(i); distract.append(c)
                break
    conv = [OPENERS[rng.randint(3)] + t.strip() + CLOSERS[rng.randint(3)] for t in texts]
    return dict(neg_texts=neg_texts, neg_idx=np.array(neg_idx), distract=np.array(distract), conv_texts=conv)


def probe_metrics(predict, labels, probes, base_pred):
    out = {}
    if len(probes["neg_idx"]):
        pn = predict(probes["neg_texts"])
        out["probe_neg_acc"] = accuracy_score(labels[probes["neg_idx"]], pn)
        out["probe_neg_pulled_to_distractor"] = float(np.mean(pn == probes["distract"]))
    pc = predict(probes["conv_texts"])
    out["probe_conv_macro_f1"] = f1_score(labels, pc, average="macro")
    return out


def latency(predict_one, texts, n, warmup):
    for t in texts[:warmup]:
        predict_one([t])
    ts = []
    for t in texts[:n]:
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter(); predict_one([t])
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    return float(np.median(ts)), float(np.percentile(ts, 90))


## 4. Validation-only hyper-parameter selection and resumable runner

In [7]:
# SKATR_SETUP 11
# ============================== MODEL REGISTRY + VALIDATION-ONLY HYPER-PARAMETER SELECTION ==============================
def spec_ft(enc):
    return dict(kind="neural", enc=enc, pooling="cls", use_lex=False, alpha=0.0, beta_s=0.0, beta_c=0.0)


def spec_sk(enc, hp):
    return dict(kind="neural", enc=enc, pooling="attn", use_lex=True, K=K_RATIONALE,
                alpha=hp["alpha"], beta_s=hp["beta_s"], beta_c=hp["beta_c"])


BN = lambda enc: BACKBONE_NAMES[enc]
OURS = f"SKAT-R v3 ({BN(MAIN_BACKBONE)})"
LLM_MODEL = f"Zero-shot LLM ({LLM_NAME.split('/')[-1]})"
ABL_DEF = {"w/o FR": dict(beta_s=0.0, beta_c=0.0), "w/o RCR": dict(alpha=0.0),
           "w/o RCR & FR": dict(alpha=0.0, beta_s=0.0, beta_c=0.0), "w/o lexical branch": dict(use_lex=False),
           "FR sufficiency only": dict(beta_c=0.0)}


def build_registry(hp):
    main = {"TF-IDF + LogReg": dict(kind="lex", clf="lr"), "TF-IDF + LinearSVM": dict(kind="lex", clf="svm")}
    if RUN_LLM:
        main[LLM_MODEL] = dict(kind="llm")
    for enc in DS_CFG["backbones"] + [e for e in DS_CFG.get("extra_ft", []) if e not in DS_CFG["backbones"]]:
        main[f"Fine-tuned {BN(enc)}"] = spec_ft(enc)
    for enc in DS_CFG["backbones"]:
        main[f"SKAT-R v3 ({BN(enc)})"] = spec_sk(enc, hp)
    abl = {k: dict(main[OURS], **ABL_DEF[k]) for k in DS_CFG["ablations"]}
    return main, abl


HP_FILE = os.path.join(OUT, "hp_selection.json")
HP_GRID_FILE = os.path.join(OUT, "hp_grid.jsonl")


def select_hyperparameters():
    """Grid over (α, β_s) on the seed-42 VALIDATION split only. Rule (stated before running):
    keep configs whose val macro-F1 is within HP_F1_TOL of the best, then take the highest val AOPC-comprehensiveness;
    ties -> smaller α, then smaller β_s. The test split is never touched here."""
    if os.path.exists(HP_FILE):
        return json.load(open(HP_FILE))
    if DS_CFG.get("hp_from"):
        cands = [os.path.join(OUT_ROOT, DS_CFG["hp_from"], "hp_selection.json")] + \
                sorted(glob.glob(f"/kaggle/input/**/skatr3_results/{DS_CFG['hp_from']}/hp_selection.json", recursive=True))
        for c in cands:
            if os.path.exists(c):
                hp = dict(json.load(open(c)), inherited_from=c)
                json.dump(hp, open(HP_FILE, "w"), indent=2)
                return hp
    d = seed_data(EXTRAS_SEED)
    done = {}
    if os.path.exists(HP_GRID_FILE):
        for r in map(json.loads, open(HP_GRID_FILE)):
            done[(r["alpha"], r["beta_s"])] = r
    rng = np.random.RandomState(0)
    vpos = rng.permutation(len(d["va"]))[:200]
    v_t = [d["texts"]["va"][j] for j in vpos]
    for a in DS_CFG["hp_grid"]["alpha"]:
        for bs_ in DS_CFG["hp_grid"]["beta_s"]:
            if (a, bs_) in done:
                continue
            hp = dict(alpha=a, beta_s=bs_, beta_c=BETA_C_RATIO * bs_)
            print(f"[HP] training config α={a}, β_s={bs_} on seed {EXTRAS_SEED} (validation only) ...", flush=True)
            t0 = time.time()
            pipe, info = train_neural(spec_sk(MAIN_BACKBONE, hp), d, EXTRAS_SEED)
            fa = eraser_aopc(pipe.predict_proba, v_t, pipe.native_scores(v_t), EXTRAS_SEED)
            r = dict(alpha=a, beta_s=bs_, beta_c=hp["beta_c"], val_macro_f1=info["val_macro_f1"],
                     val_aopc_comp=fa.get("aopc_comp", float("nan")), val_aopc_suff=fa.get("aopc_suff", float("nan")))
            append(HP_GRID_FILE, r); done[(a, bs_)] = r
            print(f"[HP] α={a} β_s={bs_}: val F1={100*r['val_macro_f1']:.2f} val AOPC-comp={r['val_aopc_comp']:.3f} ({time.time()-t0:.0f}s)")
            del pipe
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
    g = pd.DataFrame(done.values())
    ok = g[g.val_macro_f1 >= g.val_macro_f1.max() - HP_F1_TOL]
    best = ok.sort_values(["val_aopc_comp", "alpha", "beta_s"], ascending=[False, True, True]).iloc[0]
    hp = dict(alpha=float(best.alpha), beta_s=float(best.beta_s), beta_c=float(best.beta_c), rule="val-F1 within tol, max val AOPC-comp",
              grid=g.to_dict("records"))
    json.dump(hp, open(HP_FILE, "w"), indent=2)
    g.to_csv(os.path.join(OUT, "table_hp_selection.csv"), index=False)
    return hp


# ============================== RUNNER (resumable) ==============================
def done_keys():
    if not os.path.exists(RUNS_FILE):
        return set()
    return {(r["model"], r["seed"]) for r in map(json.loads, open(RUNS_FILE))}


def append(path, rec):
    with open(path, "a") as f:
        f.write(json.dumps(rec, default=float) + "\n")


def slug(s):
    return re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")


def run_one(name, spec, group, seed):
    d = seed_data(seed)
    te_t, te_y = d["texts"]["te"], d["y"]["te"]
    rec = dict(model=name, group=group, seed=seed, kind=spec["kind"], backbone=BACKBONE_NAMES.get(spec.get("enc"), "-"),
               alpha=spec.get("alpha"), beta_s=spec.get("beta_s"), beta_c=spec.get("beta_c"))
    proba = native = pipe = None
    if spec["kind"] == "lex":
        m = d["lex"][spec["clf"]]
        rec.update(val_macro_f1=d["lex_val_f1"][spec["clf"]], C=m.C_reg, explanation="coef x tfidf")
        proba, native = m.predict_proba, m.native_scores
    elif spec["kind"] == "llm":
        cache = llm_probs()
        P = np.stack([cache[int(i)] for i in d["te"]])
        rec["explanation"] = "none"
    else:
        pipe, info = train_neural(spec, d, seed)
        rec.update(info, explanation="attention" if spec["pooling"] == "attn" else "grad x input")
        proba, native = pipe.predict_proba, (lambda T, cls: pipe.native_scores(T))
    if proba is not None:
        P = proba(te_t)
    rec.update(cls_metrics(te_y, P))
    np.savez_compressed(os.path.join(OUT, "preds", f"{slug(name)}__{seed}.npz"), y=te_y, P=P.astype(np.float32), idx=d["te"])

    if proba is not None and len(d["explain_pos"]):
        t0 = time.time()
        ep = d["explain_pos"]
        Et, Ey, Ecls = [te_t[j] for j in ep], te_y[ep], P[ep].argmax(1)
        nat = native(Et, Ecls)
        occ = occlusion_scores(proba, Et, DS_CFG["max_occ_words"])
        golds = {"pmi": lexicon_gold(d["R"][K_EVAL])}
        if EXT_LEX:
            golds["ext"] = lexicon_gold(EXT_LEX)
        if HUMAN:
            gi = d["te"][ep]
            golds["human"] = lambda i, low, fl, y, gi=gi: [j in HUMAN.get(int(gi[i]), set()) for j in range(len(low))]
        for gname, gfn in golds.items():
            for ename, sc in (("native", nat), ("occ", occ)):
                r = plausibility(Et, Ey, sc, gfn)
                rec[f"rp_{gname}_{ename}"], rec[f"auprc_{gname}_{ename}"] = r["rp"], r["auprc"]
            rec[f"rp_{gname}_random"], rec[f"auprc_{gname}_random"], rec[f"n_{gname}"] = r["rp_random"], r["auprc_random"], r["n"]
        rec.update(eraser_aopc(proba, Et, nat, seed))
        rec["explain_time_s"] = time.time() - t0

    if proba is not None:
        predict = lambda T: proba(T).argmax(1)
        if DS_CFG["probes"]:
            rec.update(probe_metrics(predict, te_y, make_probes(te_t, te_y, d["R"][K_EVAL], seed), P.argmax(1)))
        if OOD is not None:
            po = predict(OOD["texts"])
            rec["ood_acc"] = accuracy_score(OOD["y"], po)
            rec["ood_macro_f1"] = f1_score(OOD["y"], po, labels=sorted(set(OOD["y"])), average="macro")
        if seed == EXTRAS_SEED and group == "main":
            ex = dict(model=name, seed=seed)
            if DEVICE == "cuda":
                ex["latency_gpu_ms_median"], ex["latency_gpu_ms_p90"] = latency(predict, te_t, LATENCY_N, LATENCY_WARMUP)
            if pipe is not None:
                globals()["DEVICE"], dev_bak = "cpu", DEVICE
                pipe.model.to("cpu")
                ex["latency_cpu_ms_median"], ex["latency_cpu_ms_p90"] = latency(predict, te_t, LATENCY_CPU_N, 3)
                globals()["DEVICE"] = dev_bak; pipe.model.to(DEVICE)
            else:
                ex["latency_cpu_ms_median"], ex["latency_cpu_ms_p90"] = latency(predict, te_t, LATENCY_N, LATENCY_WARMUP)
            append(EXTRAS_FILE, ex)
    if name == OURS and seed == EXTRAS_SEED:
        wrong = np.where(P.argmax(1) != te_y)[0]
        pd.DataFrame(dict(text_id=d["te"][wrong], text=[te_t[j] for j in wrong], true=[LABELS[te_y[j]] for j in wrong],
                          pred=[LABELS[P[j].argmax()] for j in wrong], confidence=P[wrong].max(1))) \
            .to_csv(os.path.join(OUT, "errors_ours_seed42.csv"), index=False)
    append(RUNS_FILE, rec)
    if pipe is not None:
        del pipe
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return rec


def run_all(main, abl):
    done = done_keys()
    jobs = [(n, s, "main") for n, s in main.items()] + [(n, s, "ablation") for n, s in abl.items()]
    total, k = len(jobs) * len(SEEDS), 0
    for seed in SEEDS:
        for name, spec, group in jobs:
            k += 1
            if (name, seed) in done:
                continue
            t0 = time.time()
            print(f"[{k}/{total}] seed={seed} {name}: started", flush=True)
            r = run_one(name, spec, group, seed)
            print(f"[{k}/{total}] seed={seed} {name:<34} F1={100*r['test_macro_f1']:6.2f}  ECE={100*r['test_ece']:5.2f}  "
                  f"AOPC-comp={r.get('aopc_comp', float('nan')):.3f}  RP-ext(occ)={100*r.get('rp_ext_occ', float('nan')):6.2f}  "
                  f"({time.time()-t0:5.0f}s)", flush=True)


## 5. Revision experiments: definitions (Integrated Gradients, [CLS] attention, Lin-TaSc, independent faithfulness checks)

In [8]:
# SKATR_REV_DEFS
# ============================== REVISION EXPERIMENTS: definitions ==============================
# Answers supervisor issues 2 and 3:
#   (2) stronger post hoc explainers for fine-tuned BiomedBERT: Integrated Gradients and last-layer [CLS] attention,
#       plus two faithfulness checks that FR does NOT optimise (top-1 decision flip; rank correlation with true
#       leave-one-word-out effects);
#   (3) one prior faithfulness method run on two datasets: Lin-TaSc (Chrysostomou & Aletras, ACL 2021).
# Everything is computed on the SAME splits, seeds and explanation subsets as the original study.
import traceback
from scipy.stats import spearmanr

REV_ROOT = "/kaggle/working/skatr_revision" if ON_KAGGLE else os.path.abspath("./skatr_revision")
os.makedirs(REV_ROOT, exist_ok=True)
REV_FILE = os.path.join(REV_ROOT, "revision_runs.jsonl")
if not os.path.exists(REV_FILE):          # resume: attach a previous version's output as input
    _prev = sorted(glob.glob("/kaggle/input/**/skatr_revision/revision_runs.jsonl", recursive=True))
    if _prev:
        shutil.copy(_prev[-1], REV_FILE)
        print("Resumed revision runs from", _prev[-1])

# dataset -> models to train (cheapest dataset first). "lr" (TF-IDF + LogReg) is always added; it needs no GPU.
REV_PLAN = {
    "symptom2disease":   ["ft", "skatr", "tasc"],
    "ade_corpus":        ["ft", "skatr"],
    "medical_abstracts": ["ft", "skatr", "tasc"],
}
# SKAT-R alpha selected on validation in the ORIGINAL study (Supplementary Table S5); not re-tuned here.
REV_HP_ALPHA = {"symptom2disease": 0.10, "ade_corpus": 0.50, "medical_abstracts": 0.10}
IG_STEPS = 32           # Integrated Gradients: midpoint Riemann sum, [PAD]-embedding baseline


def ig_word_scores(pipe, texts, steps=IG_STEPS, chunk=16):
    """Integrated Gradients (Sundararajan et al., 2017) w.r.t. the input word embeddings, target = predicted class.
    Baseline: [PAD] embedding at every word piece, [CLS]/[SEP] kept. Word score = |sum of word-piece attributions|."""
    model = pipe.model
    model.eval()
    emb = model.encoder.get_input_embeddings()
    pad_id = pipe.tok.pad_token_id if pipe.tok.pad_token_id is not None else 0
    alphas = (torch.arange(steps, device=DEVICE, dtype=torch.float32) + 0.5) / steps
    res, deltas = [], []
    for t in texts:
        f0 = pipe.feats([t]); f = trim(f0)
        ids = f["input_ids"].to(DEVICE); am = f["attention_mask"].to(DEVICE)
        valid = f["valid"].to(DEVICE); lex = f["lex"].to(DEVICE)
        with torch.no_grad():
            e = emb(ids).float()
            word = (f["widx"].to(DEVICE) >= 0).float().unsqueeze(-1)
            base = e * (1 - word) + emb(torch.full_like(ids, pad_id)).float() * word
            out_x, _, _ = model(inputs_embeds=e, attention_mask=am, valid=valid, lex=lex)
            out_b, _, _ = model(inputs_embeds=base, attention_mask=am, valid=valid, lex=lex)
            target = int(out_x.argmax(-1))
        total = torch.zeros_like(e[0])
        for s in range(0, steps, chunk):
            a = alphas[s:s + chunk].view(-1, 1, 1)
            x = (base + a * (e - base)).detach().requires_grad_(True)
            k = x.shape[0]
            with torch.enable_grad():
                out, _, _ = model(inputs_embeds=x, attention_mask=am.expand(k, -1), valid=valid.expand(k, -1), lex=lex.expand(k, -1))
                g = torch.autograd.grad(out[:, target].sum(), x)[0]
            total += g.sum(0).detach()
        attr = ((e - base)[0] * total / steps).sum(-1)
        # completeness check: sum of attributions should approximate f(x) - f(baseline)
        deltas.append(abs(float(attr.sum()) - float(out_x[0, target] - out_b[0, target])))
        tok = (attr * valid[0]).detach().cpu().numpy()
        res.append(np.abs(_word_agg(tok, f["widx"][0].numpy(), f0["n_words"][0])))
    ig_word_scores.last_mean_delta = float(np.mean(deltas)) if deltas else float("nan")
    return res


@torch.no_grad()
def cls_attention_word_scores(pipe, texts, bs=16):
    """Last-layer self-attention from [CLS] to every token, averaged over heads (fine-tuned model's own attention).
    Computed from the last layer's query/key projections, so it does not depend on the attention implementation."""
    model = pipe.model
    model.eval()
    enc = model.encoder
    att_mod = enc.encoder.layer[-1].attention
    sa = att_mod.self if hasattr(att_mod, "self") else att_mod
    nh = enc.config.num_attention_heads
    dh = enc.config.hidden_size // nh
    res = []
    for s in range(0, len(texts), bs):
        f0 = pipe.feats(texts[s:s + bs]); f = trim(f0)
        ids = f["input_ids"].to(DEVICE); am = f["attention_mask"].to(DEVICE)
        hs = enc(input_ids=ids, attention_mask=am, output_hidden_states=True).hidden_states[-2].float()
        B, L, _ = hs.shape
        q = sa.query(hs[:, :1]).view(B, 1, nh, dh).transpose(1, 2)
        k = sa.key(hs).view(B, L, nh, dh).transpose(1, 2)
        att = (q @ k.transpose(-1, -2)) / math.sqrt(dh)
        att = att.masked_fill(am[:, None, None, :] == 0, -1e4).softmax(-1).mean(1)[:, 0]
        tok = (att * f["valid"].to(DEVICE)).cpu().numpy()
        for i in range(B):
            res.append(_word_agg(tok[i], f["widx"][i].numpy(), f0["n_words"][i]))
    return res


class TaScModel(SKATR):
    """Lin-TaSc (Chrysostomou & Aletras, 2021), re-implemented with the same encoder, recipe and splits as SKAT-R.
    Additive attention a_t over contextual states is scaled by a learned task-specific score of the NON-contextual
    word embedding: s_t = 1 + u·e_t (u initialised at 0, so training starts from plain attention pooling).
    Pooled = Σ a_t s_t h_t; explanation = |a_t s_t|. No lexical prior, no RCR, no FR."""

    def __init__(self, enc_name, n_cls, pooling="attn", use_lex=False, dropout=0.2):
        super().__init__(enc_name, n_cls, "attn", False, dropout)
        self.tasc_u = nn.Parameter(torch.zeros(self.encoder.config.hidden_size))

    def forward(self, input_ids=None, attention_mask=None, valid=None, lex=None, inputs_embeds=None):
        if inputs_embeds is None:
            inputs_embeds = self.encoder.get_input_embeddings()(input_ids)
        hs = self.encoder(inputs_embeds=inputs_embeds, attention_mask=attention_mask).last_hidden_state
        sc = self.attn_scorer(hs).squeeze(-1).float().masked_fill(valid == 0, -1e4)
        a = torch.softmax(sc, dim=-1)
        scale = 1.0 + inputs_embeds.float() @ self.tasc_u.float()
        at = a * scale * (valid > 0).float()
        pooled = torch.bmm(at.unsqueeze(1).to(hs.dtype), hs).squeeze(1)
        ctx = self.ctx_head(self.dropout(pooled)).float()
        return ctx, at.abs(), ctx


def train_with(model_cls, spec, d, seed):
    """Run the unchanged training loop of the original study with a different model class."""
    g = globals(); orig = g["SKATR"]; g["SKATR"] = model_cls
    try:
        return train_neural(spec, d, seed)
    finally:
        g["SKATR"] = orig


def faith_independent(proba, texts, scores, occ, seed, max_words):
    """Two faithfulness checks that FR does not optimise:
    flip_top1  = share of documents whose prediction changes when the single top-ranked word is deleted
                 (flip_random: same with a random word; drop_top1: mean drop in p(ŷ));
    spearman_loo = mean Spearman correlation between the explanation and the TRUE leave-one-word-out effect
                 of each word on p(ŷ) (computed on the first max_words content words; documents with ≥3 words)."""
    rng = np.random.RandomState(seed + 1)
    P = proba(list(texts)); yhat = P.argmax(1)
    v_top, v_rnd, own, rhos = [], [], [], []
    for n, (t, s, o) in enumerate(zip(texts, scores, occ)):
        raw, low, _ = analyse(t)
        content = [i for i, w in enumerate(low) if w not in PUNCT]
        if len(content) < 2:
            continue
        s = np.asarray(s, float)
        sc = np.array([s[i] if i < len(s) else 0.0 for i in content])
        top = content[int(np.argmax(sc))]
        rnd = content[rng.randint(len(content))]
        v_top.append(" ".join(raw[:top] + raw[top + 1:]) or ".")
        v_rnd.append(" ".join(raw[:rnd] + raw[rnd + 1:]) or ".")
        own.append(n)
        cw = content[:max_words]
        a = np.array([s[i] if i < len(s) else 0.0 for i in cw])
        b = np.array([o[i] if i < len(o) else 0.0 for i in cw])
        if len(cw) >= 3 and a.std() > 0 and b.std() > 0:
            rhos.append(spearmanr(a, b).correlation)
    if not own:
        return {}
    own = np.array(own)
    Qt, Qr = proba(v_top), proba(v_rnd)
    yh = yhat[own]
    return dict(flip_top1=float(np.mean(Qt.argmax(1) != yh)), flip_random=float(np.mean(Qr.argmax(1) != yh)),
                drop_top1=float(np.mean(P[own, yh] - Qt[np.arange(len(own)), yh])),
                spearman_loo=float(np.mean(rhos)) if rhos else float("nan"), n_spearman=len(rhos))


def rev_done():
    if not os.path.exists(REV_FILE):
        return set()
    return {(r["dataset"], r["seed"], r["model"]) for r in map(json.loads, open(REV_FILE))}


def run_revision_dataset():
    plan = REV_PLAN[DATASET]
    hp = dict(alpha=REV_HP_ALPHA[DATASET], beta_s=1.0, beta_c=BETA_C_RATIO * 1.0)
    specs = {
        "ft":    ("Fine-tuned BiomedBERT", spec_ft(BIOMED), None),
        "skatr": ("SKAT-R (BiomedBERT)", spec_sk(BIOMED, hp), None),
        "tasc":  ("TaSc (BiomedBERT)", dict(kind="neural", enc=BIOMED, pooling="attn", use_lex=False,
                                            alpha=0.0, beta_s=0.0, beta_c=0.0), TaScModel),
    }
    jobs = [("lr", "TF-IDF + LogReg", None, None)] + [(k,) + specs[k] for k in plan]
    for seed in SEEDS:
        d = seed_data(seed)
        ep = d["explain_pos"]
        te_t, te_y = d["texts"]["te"], d["y"]["te"]
        Et, Ey = [te_t[j] for j in ep], te_y[ep]
        golds = {}
        if EXT_LEX:
            golds["ext"] = lexicon_gold(EXT_LEX)
        if HUMAN:
            gi = d["te"][ep]
            golds["human"] = lambda i, low, fl, y, gi=gi: [j in HUMAN.get(int(gi[i]), set()) for j in range(len(low))]
        for key, name, spec, cls in jobs:
            if (DATASET, seed, name) in rev_done():
                continue
            if _rev_left_h() < 0.4:
                raise TimeBudgetReached()
            t0 = time.time()
            print(f"[revision] {DATASET} seed={seed} {name}: started", flush=True)
            info, pipe = {}, None
            if key == "lr":
                m = d["lex"]["lr"]
                proba = m.predict_proba
                expl = {"coef x tfidf": m.native_scores(Et, m.predict_proba(Et).argmax(1))}
            else:
                pipe, info = train_with(cls, spec, d, seed) if cls is not None else train_neural(spec, d, seed)
                proba = pipe.predict_proba
                if key == "ft":
                    expl = {"grad x input": pipe.grad_word_scores(Et)}
                    for ename, fn in (("integrated gradients", ig_word_scores), ("CLS attention", cls_attention_word_scores)):
                        try:                                    # one failing explainer must not lose the others
                            expl[ename] = fn(pipe, Et)
                        except Exception:
                            print(f"    [warning] {ename} failed and is skipped:"); traceback.print_exc()
                    if "integrated gradients" in expl:
                        info["ig_mean_completeness_gap"] = ig_word_scores.last_mean_delta
                else:
                    expl = {("attention" if key == "skatr" else "scaled attention (TaSc)"): pipe.attention_word_scores(Et)}
            P = proba(te_t)
            base = dict(dataset=DATASET, seed=seed, model=name, n_tr=len(d["tr"]), n_te=len(d["te"]),
                        fixed_test=bool(DS_CFG["fixed_test"]), n_explain=len(Et), **cls_metrics(te_y, P),
                        **{k: v for k, v in info.items() if isinstance(v, (int, float))})
            occ = occlusion_scores(proba, Et, DS_CFG["max_occ_words"])
            rows = []
            for ename, sc in expl.items():
                r = dict(base, explanation=ename)
                r.update(eraser_aopc(proba, Et, sc, seed))
                r.update(faith_independent(proba, Et, sc, occ, seed, DS_CFG["max_occ_words"]))
                for gname, gfn in golds.items():
                    pl = plausibility(Et, Ey, sc, gfn)
                    r[f"rp_{gname}"], r[f"auprc_{gname}"], r[f"rp_{gname}_random"] = pl["rp"], pl["auprc"], pl["rp_random"]
                rows.append(r)
            with open(REV_FILE, "a") as fh:            # all explanations of one model-seed are written together
                for r in rows:
                    fh.write(json.dumps(r, default=float) + "\n")
            for r in rows:
                print(f"    {r['explanation']:<26} F1={100*r['test_macro_f1']:5.1f}  comp={100*r.get('aopc_comp', np.nan):5.1f}  "
                      f"suff={100*r.get('aopc_suff', np.nan):5.1f}  flip@1={100*r.get('flip_top1', np.nan):5.1f}  "
                      f"rho_LOO={r.get('spearman_loo', np.nan):.3f}", flush=True)
            print(f"[revision] {DATASET} seed={seed} {name}: done in {time.time()-t0:.0f}s", flush=True)
            if pipe is not None:
                del pipe
                if DEVICE == "cuda":
                    torch.cuda.empty_cache()


## 6. Run (all datasets in one session; time-guarded and resumable)

In [9]:
# REVISION_DRIVER
# ============================== RUN ALL REVISION EXPERIMENTS (time-guarded, resumable) ==============================
import traceback, gc
try:
    import psutil
    _T0 = psutil.Process(os.getpid()).create_time()      # kernel start = start of this Kaggle session
except Exception:
    _T0 = time.time()
BUDGET_H = 11.0                                          # Kaggle stops sessions at 12 h


class TimeBudgetReached(Exception):
    pass


def _rev_left_h():
    return BUDGET_H - (time.time() - _T0) / 3600


_setup = {}
for _c in In:
    _m = re.match(r"\s*# SKATR_SETUP (\d+)", _c)
    if _m:
        _setup[int(_m.group(1))] = _c                    # last executed version of each setup cell
_setup = [_setup[k] for k in sorted(_setup)]
if not _setup:
    raise RuntimeError("Run the notebook from the top (Save Version -> Save & Run All).")

_stopped = False
for _ds in REV_PLAN:
    if _rev_left_h() < 0.75:
        print(f"[driver] not starting {_ds}: under 45 min left. Save the version, attach its output and re-run.", flush=True)
        _stopped = True
        break
    print("\n" + "=" * 30, "DATASET:", _ds, f"({_rev_left_h():.2f} h left)", "=" * 30, flush=True)
    try:
        for _src in _setup:
            exec(_src.replace('DATASET = "symptom2disease"', f'DATASET = "{_ds}"'), globals())
        run_revision_dataset()
    except TimeBudgetReached:
        print(f"[driver] time budget reached during {_ds}; finished runs are saved.", flush=True)
        _stopped = True
        break
    except Exception:
        print(f"[driver] ERROR in {_ds}; continuing with the next dataset", flush=True)
        traceback.print_exc()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print("[driver] done." if not _stopped else "[driver] stopped early: re-run with this version's output attached.", flush=True)



============================== DATASET: symptom2disease (10.99 h left) ==============================
Dataset: symptom2disease | output: /kaggle/working/skatr_revision/base/symptom2disease
{
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "transformers": "5.20.2",
  "sklearn": "1.6.1",
  "device": "cuda",
  "gpu": "Tesla T4",
  "cpu": "x86_64",
  "cpu_count": 4,
  "cfg": {
    "max_len": 128,
    "batch_size": 16,
    "epochs": 8,
    "patience": 3,
    "lr_encoder": 3e-05,
    "lr_head": 0.001,
    "weight_decay": 0.01,
    "warmup_frac": 0.1,
    "dropout": 0.2,
    "grad_clip": 1.0
  },
  "seeds": [
    42,
    123,
    456,
    789,
    2024
  ],
  "main_backbone": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
  "backbones": [
    "distilbert-base-uncased",
    "emilyalsentzer/Bio_ClinicalBERT",
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
  ],
  "extra_ft": [],
  "llm": null,
  "fixed": {
    "K_RATIONALE": 10,
    "K_EVAL": 10,
    "

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

... (137 outputs left out)

    epoch 1/4: val macro-F1=81.60  (256s elapsed)
    epoch 2/4: val macro-F1=83.54  (510s elapsed)
    epoch 3/4: val macro-F1=85.42  (764s elapsed)
    epoch 4/4: val macro-F1=84.96  (1017s elapsed)
    attention                  F1= 84.9  comp= 43.9  suff=  5.2  flip@1=  3.3  rho_LOO=0.116
[revision] medical_abstracts seed=2024 SKAT-R (BiomedBERT): done in 1126s
[revision] medical_abstracts seed=2024 TaSc (BiomedBERT): started


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    epoch 1/4: val macro-F1=82.76  (99s elapsed)
    epoch 2/4: val macro-F1=86.31  (198s elapsed)
    epoch 3/4: val macro-F1=88.48  (296s elapsed)
    epoch 4/4: val macro-F1=87.79  (394s elapsed)
    scaled attention (TaSc)    F1= 86.1  comp= 29.2  suff=  9.9  flip@1=  1.3  rho_LOO=0.052
[revision] medical_abstracts seed=2024 TaSc (BiomedBERT): done in 493s
[driver] done.


## 7. Results: means over seeds, SKAT-R vs every explanation (corrected resampled t-test, Holm), zip for download

In [10]:
# REVISION_SUMMARY
# ============================== SUMMARY TABLES + STATISTICS (corrected resampled t-test, Holm) ==============================
from scipy import stats as _st
rev = pd.DataFrame([json.loads(l) for l in open(REV_FILE)])
rev = rev.drop_duplicates(["dataset", "seed", "model", "explanation"], keep="last")
rev["method"] = rev["model"] + " | " + rev["explanation"]
REF = "SKAT-R (BiomedBERT) | attention"
METRICS = [("test_macro_f1", "higher"), ("test_ece", "lower"), ("aopc_comp", "higher"), ("aopc_suff", "lower"),
           ("flip_top1", "higher"), ("drop_top1", "higher"), ("spearman_loo", "higher"),
           ("rp_human", "higher"), ("auprc_human", "higher"), ("rp_ext", "higher"), ("auprc_ext", "higher")]
METRICS = [(m, d) for m, d in METRICS if m in rev.columns]

summ = rev.groupby(["dataset", "method"])[[m for m, _ in METRICS] + ["flip_random"]].agg(["mean", "std"])
summ.to_csv(os.path.join(REV_ROOT, "revision_summary.csv"))
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
print("Seeds per method:\n", rev.groupby(["dataset", "method"]).seed.nunique().to_string())


def _ctest(a, b, ratio, fixed):
    dlt = np.asarray(a, float) - np.asarray(b, float)
    J = len(dlt)
    if J < 2:
        return float(dlt.mean()), np.nan, np.nan, np.nan
    se = math.sqrt(dlt.var(ddof=1) * ((1 / J) if fixed else (1 / J + ratio)))
    if se == 0:
        return float(dlt.mean()), float(dlt.mean()), float(dlt.mean()), (0.0 if dlt.mean() != 0 else 1.0)
    t = dlt.mean() / se; q = _st.t.ppf(0.975, J - 1)
    return float(dlt.mean()), float(dlt.mean() - q * se), float(dlt.mean() + q * se), float(2 * _st.t.sf(abs(t), J - 1))


def _holm(p):
    p = np.asarray(p, float); out = np.full(len(p), np.nan); ok = ~np.isnan(p)
    idx = np.where(ok)[0][np.argsort(p[ok])]; m = len(idx); run = 0.0
    for r, i in enumerate(idx):
        run = max(run, min(1.0, (m - r) * p[i])); out[i] = run
    return out


rows = []
for ds, g in rev.groupby("dataset"):
    ratio = float((g.n_te / g.n_tr).iloc[0]); fixed = bool(g.fixed_test.iloc[0])
    ref = g[g.method == REF].set_index("seed")
    if ref.empty:
        continue
    for met, direction in METRICS:
        fam = []
        for meth, h in g[g.method != REF].groupby("method"):
            h = h.set_index("seed")
            s = sorted(set(ref.index) & set(h.index))
            if met not in h or h.loc[s, met].isna().all() or ref.loc[s, met].isna().all():
                continue
            diff, lo, hi, p = _ctest(ref.loc[s, met], h.loc[s, met], ratio, fixed)
            fam.append(dict(dataset=ds, metric=met, better=direction, comparison=f"{REF}  vs  {meth}", n_seeds=len(s),
                            diff_pts=100 * diff if met != "spearman_loo" else diff,
                            ci_low=100 * lo if met != "spearman_loo" else lo, ci_high=100 * hi if met != "spearman_loo" else hi, p=p))
        if fam:
            for r, ph in zip(fam, _holm([r["p"] for r in fam])):
                r["p_holm"], r["family_size"] = ph, len(fam)
            rows += fam
stats_df = pd.DataFrame(rows)
stats_df.to_csv(os.path.join(REV_ROOT, "revision_statistics.csv"), index=False)

print("\n=== Means over seeds (percent; spearman_loo as correlation) ===")
show = rev.groupby(["dataset", "method"])[[m for m, _ in METRICS] + ["flip_random"]].mean()
for c in show.columns:
    if c != "spearman_loo":
        show[c] = 100 * show[c]
print(show.round(1).to_string())
print("\n=== SKAT-R attention vs every other explanation (diff = SKAT-R minus other; Holm within dataset x metric) ===")
if len(stats_df):
    print(stats_df.round(4).to_string(index=False))
shutil.make_archive(os.path.join("/kaggle/working" if ON_KAGGLE else ".", "skatr_revision_results"), "zip", REV_ROOT)
print("\nSaved:", REV_FILE, "| revision_summary.csv | revision_statistics.csv | skatr_revision_results.zip")


Seeds per method:
 dataset            method                                      
ade_corpus         Fine-tuned BiomedBERT | CLS attention           5
                   Fine-tuned BiomedBERT | grad x input            5
                   Fine-tuned BiomedBERT | integrated gradients    5
                   SKAT-R (BiomedBERT) | attention                 5
                   TF-IDF + LogReg | coef x tfidf                  5
medical_abstracts  Fine-tuned BiomedBERT | CLS attention           5
                   Fine-tuned BiomedBERT | grad x input            5
                   Fine-tuned BiomedBERT | integrated gradients    5
                   SKAT-R (BiomedBERT) | attention                 5
                   TF-IDF + LogReg | coef x tfidf                  5
                   TaSc (BiomedBERT) | scaled attention (TaSc)     5
symptom2disease    Fine-tuned BiomedBERT | CLS attention           5
                   Fine-tuned BiomedBERT | grad x input            5
                   F